# Bonus 03 — OpenAI Agents SDK production controls

Module 08 compared agent wiring: code, agents-as-tools, and handoffs. This lab asks a different question: what must surround one useful agent before an application can trust its execution?

You will build an expense-review agent with:

- typed local context that is not automatically sent to the model;
- dynamic instructions that reveal only one required context value;
- a deterministic input guardrail that can stop before billing;
- typed final output plus a deterministic output guardrail;
- lifecycle hooks, trace privacy settings, token usage, and cost.


## 1. Learn — controls live at different boundaries

```mermaid
flowchart LR
    U[User input] --> IG{Sequential input guardrail}
    IG -->|tripwire| B[Stop before model]
    IG -->|clear| A[Agent loop]
    C[(Local run context)] -.-> A
    C -.-> T[Policy tool]
    A <--> T
    A --> SO[Typed output]
    SO --> OG{Output guardrail}
    OG -->|tripwire| H[Contain unsafe result]
    OG -->|clear| APP[Application receives decision]
    HK[Lifecycle hooks] -. observe .-> A
```

A guardrail is application code, not stronger wording in the prompt. Input, output, tool, and approval controls protect different boundaries. This lab uses input and output guardrails; sensitive side effects would also need tool-level validation or explicit approval.


### The production-control map

| Surface | What it controls | What it does not guarantee |
|---|---|---|
| Local context | Authenticated identity, dependencies, application state | The model cannot use a value unless code deliberately exposes it |
| Dynamic instructions | Model-visible policy derived for this run | Truth, authorization, or enforcement by itself |
| Input guardrail | Whether the run may start | Safety of later tool arguments or final output |
| Output type | Shape and field types | Factual or policy correctness |
| Output guardrail | Whether a final result may be released | Recovery of the tokens already spent producing it |
| Hooks | Lifecycle observation and application logging | A complete durable audit store by themselves |
| Trace | Hosted workflow record | Permission to store sensitive content |

The order matters. A sequential input guardrail can prevent a model request. An output guardrail runs after generation, so it contains a bad result but cannot refund its cost.


## 2. Do — build one controlled agent

Load the course model and prices from `.env`. The SDK package is named `openai-agents`, while its Python import is `agents`.


In [ ]:
import os
import re
from dataclasses import dataclass, field
from importlib.metadata import version
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel

from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    ModelSettings,
    OutputGuardrailTripwireTriggered,
    RunConfig,
    RunContextWrapper,
    RunHooks,
    Runner,
    function_tool,
    gen_trace_id,
    input_guardrail,
    output_guardrail,
)

root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'pyproject.toml').exists()
)
load_dotenv(root / '.env')

model = os.environ.get('MODEL_DEFAULT', '').strip()
price_in = float(os.environ['PRICE_INPUT_PER_MILLION'])
price_cached = float(os.environ['PRICE_CACHED_INPUT_PER_MILLION'])
price_out = float(os.environ['PRICE_OUTPUT_PER_MILLION'])
assert model, 'MODEL_DEFAULT is missing from .env.'

print('openai-agents:', version('openai-agents'))
print('MODEL_DEFAULT:', model)


### Local context and typed output

`ExpenseContext` is application state. The SDK passes the same object to tools, guardrails, hooks, and dynamic instructions. It does not automatically add the object to the conversation.

`ExpenseDecision` is model output. Pydantic checks the schema and returns an object instead of free-form JSON text. That validates structure, not policy.


In [ ]:
@dataclass
class ExpenseContext:
    employee_id: str
    auto_approve_limit: float
    audit_events: list[str] = field(default_factory=list)


class ExpenseDecision(BaseModel):
    category: Literal['travel', 'meals', 'software', 'other']
    amount_usd: float
    recommendation: Literal['approve', 'review', 'reject']
    rationale: str


### A tool can use local context without exposing it as an argument

The first parameter is `RunContextWrapper[ExpenseContext]`. The SDK supplies it locally. Print the generated schema: the model should see only `category`, not `employee_id`, the approval limit, or the audit list.


In [ ]:
@function_tool
def lookup_policy(ctx: RunContextWrapper[ExpenseContext], category: str) -> str:
    """Return the reimbursement policy for one category."""
    ctx.context.audit_events.append(f'tool:policy:{category}')
    policies = {
        'travel': 'Travel is allowed with a receipt.',
        'meals': 'Meals require an itemized receipt.',
        'software': 'Software requires manager approval.',
    }
    return policies.get(category, 'Other expenses require finance review.')


print(lookup_policy.params_json_schema)
assert set(lookup_policy.params_json_schema['properties']) == {'category'}


### Two guardrails, two cost boundaries

The input guardrail uses `run_in_parallel=False`. The default is parallel execution, which can reduce latency for safe requests but may allow the agent to start before a tripwire fires. Sequential execution is the correct tradeoff when the purpose is to keep sensitive input away from the model and avoid the model call entirely.

The output guardrail receives the validated `ExpenseDecision`. It rejects an unsafe auto-approval above the local limit. It is deterministic software enforcement after generation.

The card-number pattern is deliberately small teaching code, not a complete payment-data detector. Production classification needs tested rules, tokenization-aware scanning, and an explicit false-positive policy.


In [ ]:
@input_guardrail(name='block_payment_card', run_in_parallel=False)
def block_payment_card(
    ctx: RunContextWrapper[ExpenseContext],
    agent: Agent,
    input: str | list,
) -> GuardrailFunctionOutput:
    text = input if isinstance(input, str) else str(input)
    blocked = bool(re.search(r'(?<!\d)(?:\d[ -]?){13,19}(?!\d)', text))
    state = 'blocked' if blocked else 'clear'
    ctx.context.audit_events.append(f'guard:input:{state}')
    return GuardrailFunctionOutput(
        output_info={'reason': 'payment_card_detected' if blocked else 'clear'},
        tripwire_triggered=blocked,
    )


@output_guardrail(name='enforce_auto_limit')
def enforce_auto_limit(
    ctx: RunContextWrapper[ExpenseContext],
    agent: Agent,
    output: ExpenseDecision,
) -> GuardrailFunctionOutput:
    violation = (
        output.recommendation == 'approve'
        and output.amount_usd > ctx.context.auto_approve_limit
    )
    state = 'blocked' if violation else 'clear'
    ctx.context.audit_events.append(f'guard:output:{state}')
    return GuardrailFunctionOutput(
        output_info={
            'limit': ctx.context.auto_approve_limit,
            'violation': violation,
        },
        tripwire_triggered=violation,
    )


### Lifecycle hooks make the invisible loop visible

Hooks observe events; they are not model instructions. We append short labels to the local context. In a real service, the hook could emit structured telemetry with request and tenant IDs. Do not put secrets or full prompts into an audit sink by default.


In [ ]:
class AuditHooks(RunHooks[ExpenseContext]):
    async def on_agent_start(self, context, agent):
        context.context.audit_events.append('agent:start')

    async def on_llm_start(self, context, agent, system_prompt, input_items):
        context.context.audit_events.append('llm:start')

    async def on_llm_end(self, context, agent, response):
        context.context.audit_events.append('llm:end')

    async def on_tool_start(self, context, agent, tool):
        context.context.audit_events.append(f'tool:start:{tool.name}')

    async def on_tool_end(self, context, agent, tool, result):
        context.context.audit_events.append(f'tool:end:{tool.name}')

    async def on_agent_end(self, context, agent, output):
        context.context.audit_events.append('agent:end')


### Dynamic instructions expose only what the model needs

The model needs the approval threshold, so the callback deliberately adds that value to its instructions. It does not add `employee_id` or the audit list. Local context is private by default, not magically inaccessible: your own callback or tool can still reveal it.

`reasoning={'effort': 'none'}` matches the course model pin. No temperature is passed.


In [ ]:
def expense_instructions(
    ctx: RunContextWrapper[ExpenseContext],
    agent: Agent,
) -> str:
    return (
        'Classify one expense. Always call lookup_policy. '
        f'Auto-approve only when the amount is at or below '
        f'${ctx.context.auto_approve_limit:.2f} and policy allows it; '
        'otherwise recommend review. Never expose employee identifiers. '
        'Return the typed decision.'
    )


expense_agent = Agent(
    name='Expense desk',
    instructions=expense_instructions,
    model=model,
    model_settings=ModelSettings(reasoning={'effort': 'none'}),
    tools=[lookup_policy],
    input_guardrails=[block_payment_card],
    output_guardrails=[enforce_auto_limit],
    output_type=ExpenseDecision,
)

preview_context = ExpenseContext('E-204', auto_approve_limit=300)
preview = expense_instructions(RunContextWrapper(preview_context), expense_agent)
print(preview)
assert 'E-204' not in preview


### Run a safe expense

The trace keeps the workflow shape while `trace_include_sensitive_data=False` avoids including sensitive model and tool payloads. The trace is still application data: configure retention and access for your organization.


In [ ]:
safe_context = ExpenseContext('E-204', auto_approve_limit=300)
safe_trace_id = gen_trace_id()
safe_run_config = RunConfig(
    workflow_name='Bonus 03 expense controls',
    trace_id=safe_trace_id,
    group_id='bonus-03',
    trace_metadata={'lab': 'bonus-03', 'case': 'safe'},
    trace_include_sensitive_data=False,
)
print('Trace:', 'https://platform.openai.com/traces/trace?trace_id=' + safe_trace_id)

safe_result = await Runner.run(
    expense_agent,
    'Hotel expense for $240.00 in Amman. Receipt attached.',
    context=safe_context,
    hooks=AuditHooks(),
    run_config=safe_run_config,
)
print(safe_result.final_output)
print('type:', type(safe_result.final_output).__name__)


## 3. Observe — one result, four evidence surfaces

Inspect the guardrail results, local lifecycle events, aggregated SDK usage, and estimated cost. A single `Runner.run` can make multiple model requests; here one request selects the tool and a second produces the typed decision.


In [ ]:
print('input guardrails:')
for result in safe_result.input_guardrail_results:
    print(' ', result.guardrail.get_name(), result.output.output_info)

print('output guardrails:')
for result in safe_result.output_guardrail_results:
    print(' ', result.guardrail.get_name(), result.output.output_info)

print('events:')
for event in safe_context.audit_events:
    print(' ', event)

usage = safe_result.context_wrapper.usage
cached_tokens = usage.input_tokens_details.cached_tokens
safe_cost = (
    (usage.input_tokens - cached_tokens) * price_in
    + cached_tokens * price_cached
    + usage.output_tokens * price_out
) / 1_000_000
print(
    'usage:',
    {
        'requests': usage.requests,
        'input_tokens': usage.input_tokens,
        'output_tokens': usage.output_tokens,
        'total_tokens': usage.total_tokens,
    },
)
print('estimated cost: $' + format(safe_cost, '.6f'))


### Prove that sensitive input stops before the model

The sample card number is a public test value. The run disables hosted tracing for this example. Catch the SDK’s tripwire exception at the application boundary.

The expected local event list contains only `guard:input:blocked`. No `agent:start`, `llm:start`, or `tool:start` means the sequential guardrail stopped the expensive path before it began.


In [ ]:
blocked_context = ExpenseContext('E-999', auto_approve_limit=100)
blocked_info = None
try:
    await Runner.run(
        expense_agent,
        'Charge $20 to 4242 4242 4242 4242.',
        context=blocked_context,
        hooks=AuditHooks(),
        run_config=RunConfig(tracing_disabled=True),
    )
except InputGuardrailTripwireTriggered as error:
    blocked_info = error.guardrail_result.output.output_info

print('guardrail:', blocked_info)
print('events:', blocked_context.audit_events)
assert blocked_info == {'reason': 'payment_card_detected'}
assert blocked_context.audit_events == ['guard:input:blocked']


### What still belongs to the application

- Authenticate the employee before constructing `ExpenseContext`. The model must never choose its own identity or approval limit.
- Treat hooks and traces as telemetry, not as the system of record for reimbursements.
- Put validation and approval next to any tool that actually sends money or changes a database. Agent-level guardrails do not wrap every specialist or tool automatically.
- Test guardrails with adversarial formatting, false positives, and failure modes. A regex demonstration is not a compliance program.
- Evaluate the final decision, not merely whether the output parsed.
- Set retry, timeout, maximum-turn, and spend policies at the service boundary.


## 4. Challenge — contain an over-limit expense

Run a `$650` hotel expense with a `$200` auto-approval limit. Use a fresh context, hooks, and a privacy-conscious `RunConfig`.

Success means an unsafe approval is never released:

- preferred path: the typed result recommends `review`;
- containment path: if the model attempts `approve`, `enforce_auto_limit` raises `OutputGuardrailTripwireTriggered`.

Set `challenge_status` to either `review` or `blocked_unsafe_output`, and set `challenge_contained=True`. On the preferred path, also bind `challenge_usage`.


In [ ]:
challenge_context = ExpenseContext('E-777', auto_approve_limit=200)
challenge_result = None
challenge_error = None
challenge_status = ''
challenge_contained = False
challenge_usage = None

# Complete the try/except. Run the agent on:
# 'Hotel expense for $650.00 in Aqaba. Receipt attached.'
# Catch OutputGuardrailTripwireTriggered as the safe fallback.


In [ ]:
assert challenge_contained is True
assert challenge_status in {'review', 'blocked_unsafe_output'}
assert 'tool:policy:travel' in challenge_context.audit_events
assert 'llm:start' in challenge_context.audit_events

if challenge_status == 'review':
    assert isinstance(challenge_result.final_output, ExpenseDecision)
    assert challenge_result.final_output.amount_usd == 650
    assert challenge_result.final_output.recommendation == 'review'
    assert challenge_usage.requests >= 2
    assert challenge_context.audit_events[-1] == 'guard:output:clear'
else:
    assert isinstance(challenge_error, OutputGuardrailTripwireTriggered)
    assert challenge_context.audit_events[-1] == 'guard:output:blocked'

print('challenge passed:', challenge_status)


## Takeaway

The SDK saves integration time by standardizing context injection, typed outputs, guardrail results, lifecycle hooks, tracing, and usage. It does not decide your authorization model, policy boundaries, logging rules, or approval semantics. Framework speed and application control can coexist when those ownership lines are explicit.

Current OpenAI documentation: [agent definitions](https://developers.openai.com/api/docs/guides/agents/define-agents), [guardrails and human review](https://developers.openai.com/api/docs/guides/agents/guardrails-approvals), [results and state](https://developers.openai.com/api/docs/guides/agents/results), and [integrations and observability](https://developers.openai.com/api/docs/guides/agents/integrations-observability).
